##### HIGH LEVEL IMPLEMENTATION OF THE PAPER ' Attention Is All You Need (Vaswani et al., 2017)' USING PYTORCH

here is the link to the paper: https://arxiv.org/pdf/1706.03762

#### things to try
- use the revised mathematical expression of the positional encoding formula
- use optimized method for multi-head attention
- optimize any other implementation in code

In [43]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import torch.nn.functional as F
import re
import math
import time


In [ ]:
# amm using this for colab implemenation: uncomment this section to download the data
# import urllib.request
# import zipfile
# import os

# # Download a small, clean dataset of English-German pairs
# url = "https://www.manythings.org/anki/deu-eng.zip"
# zip_path = "deu-eng.zip"

# # Create a request with a User-Agent header to avoid HTTP Error 406
# req = urllib.request.Request(
#     url,
#     headers={
#         "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
#     },
# )

# # Open the URL and retrieve the data
# with urllib.request.urlopen(req) as response:
#     with open(zip_path, 'wb') as out_file:
#         out_file.write(response.read())

# # Extract it
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(".")

In [45]:
with open("../deu.txt", "r", encoding="utf-8") as f:
    lines = f.read().strip().split("\n")


pairs = []

eng_sentences = []
german_sentences = []


for line in lines[:3]:
    parts = line.split("\t")

    if len(parts) < 2:
        continue

    english = parts[0]
    german = parts[1]

    pairs.append((english, german))

    eng_sentences.append(english)
    german_sentences.append(german)


print(f"Number of sentence pairs: {len(pairs)}")

print(f"Sample pair: {pairs[:3]}")

Number of sentence pairs: 3
Sample pair: [('Go.', 'Geh.'), ('Hi.', 'Hallo!'), ('Hi.', 'Grüß Gott!')]


In [46]:
class VocabularyBuilder:
    def __init__(self):

        self.word2idx = {
            "<PAD>": 0,
            "<UNK>": 1,
            "<SOS>": 2,
            "<EOS>": 3,
            "</w>": 4,
        }

        self.idx2word = {
            0: "<PAD>",
            1: "<UNK>",
            2: "<SOS>",
            3: "<EOS>",
            4: "</w>",
        }

        self.merge_order = []

        self.num_words = len(self.word2idx)

    def preprocess(self, sentence):
        print("--------------------------------------- starting preprocessing....")
        print("sentence: ", sentence)
        sentence = sentence.lower()

        # Separate punctuation from words.

        sentence = re.sub(r'([.,!?;:"\'()])', r" \1 ", sentence)
        print("sentence lower cased and punctuation separated:\n", sentence)
        words = sentence.strip().split()
        print("sentence as words: \n", words)
        return words

    def preprocess2(self, all_sentences):
        print(
            "---------------------------------------- starting preprocessing all sentences"
        )

        print("_ _ _sentences before preprocessing: \n")
        print(all_sentences[:3])

        list_sentence = []

        for sentence in all_sentences:
            words = self.preprocess(sentence)

            current_sentence = []

            for word in words:
                word_list = list(word)

                # Word boundary marker
                word_list.append("</w>")

                current_sentence.append(word_list)

            list_sentence.append(current_sentence)
        print("_ _ _sentences after preprocessing: \n")
        print(list_sentence[:3])
        return list_sentence

    def build_initial_vocabulary(self, list_sentence):
        print("-----------------------starting to build initial vocabulary")

        for sentence in list_sentence:
            for word in sentence:
                for token in word:
                    if token not in self.word2idx:
                        self.word2idx[token] = self.num_words

                        self.idx2word[self.num_words] = token

                        self.num_words += 1

    def tokenize2(self, list_sentence):

        tokens = Counter()

        for sentence in list_sentence:
            for word in sentence:
                for index in range(len(word) - 1):
                    pair = (word[index], word[index + 1])

                    tokens[pair] += 1

        return tokens

    def _word_tokenizer(self, word, compare_key):

        temp_word = []

        index = 0

        while index < len(word):
            # Check whether a pair exists
            if index + 1 < len(word):
                letters = (word[index], word[index + 1])

                if letters == compare_key:
                    merged = word[index] + word[index + 1]

                    temp_word.append(merged)

                    index += 2

                    continue

            # No merge
            temp_word.append(word[index])

            index += 1

        return temp_word

    def update_vocab(self, tokens, list_sentence):
        if not tokens:
            return list_sentence

        max_key, max_value = tokens.most_common(1)[0]

        merged_token = max_key[0] + max_key[1]

        if merged_token not in self.word2idx:
            self.word2idx[merged_token] = self.num_words

            self.idx2word[self.num_words] = merged_token

            self.num_words += 1

        self.merge_order.append(max_key)

        new_sentence_list = []

        for sentence in list_sentence:
            current_sentence = []

            for word in sentence:
                temp_word = self._word_tokenizer(word, max_key)

                current_sentence.append(temp_word)

            new_sentence_list.append(current_sentence)

        return new_sentence_list

    def train(self, all_sentences, num_merges=5000):

        # Character-level representation
        list_sentence = self.preprocess2(all_sentences)

        # Initial vocabulary
        self.build_initial_vocabulary(list_sentence)

        print(f"Initial vocabulary size: {self.num_words}")

        for i in range(num_merges):
            tokens = self.tokenize2(list_sentence)

            if not tokens:
                print(f"Stopped at merge {i}: no more pairs.")

                break

            list_sentence = self.update_vocab(tokens, list_sentence)

            # Optional progress information
            if (i + 1) % 500 == 0:
                print(f"Completed {i + 1} merges")

        print(f"Final vocabulary size: {self.num_words}")

        print(f"Total merges learned: {len(self.merge_order)}")

    def numericalize(self, sentence):

        words = self.preprocess(sentence)

        num_sentence = []

        for word in words:
            # Start with characters
            word_list = list(word)

            # Add word boundary
            word_list.append("</w>")

            for key in self.merge_order:
                word_list = self._word_tokenizer(word_list, key)

            for token in word_list:
                token_id = self.word2idx.get(token, self.word2idx["<UNK>"])

                num_sentence.append(token_id)

        return num_sentence

    def tokenize(self, sentence):

        words = self.preprocess(sentence)

        all_tokens = []

        for word in words:
            word_list = list(word)

            word_list.append("</w>")

            for key in self.merge_order:
                word_list = self._word_tokenizer(word_list, key)

            all_tokens.extend(word_list)

        return all_tokens

    def decode(self, token_ids):

        tokens = []

        for token_id in token_ids:
            token = self.idx2word.get(token_id, "<UNK>")

            # Ignore special tokens
            if token in {
                "<PAD>",
                "<SOS>",
                "<EOS>",
            }:
                continue

            tokens.append(token)

        text = ""

        for token in tokens:
            if token.endswith("</w>"):
                # Remove word boundary
                token = token[:-4]

                text += token + " "

            else:
                text += token

        return text.strip()


In [47]:
tokenizer = VocabularyBuilder()

all_sentences = eng_sentences + german_sentences

tokenizer.train(all_sentences, num_merges=5000)

---------------------------------------- starting preprocessing all sentences
_ _ _sentences before preprocessing: 

['Go.', 'Hi.', 'Hi.']
--------------------------------------- starting preprocessing....
sentence:  Go.
sentence lower cased and punctuation separated:
 go . 
sentence as words: 
 ['go', '.']
--------------------------------------- starting preprocessing....
sentence:  Hi.
sentence lower cased and punctuation separated:
 hi . 
sentence as words: 
 ['hi', '.']
--------------------------------------- starting preprocessing....
sentence:  Hi.
sentence lower cased and punctuation separated:
 hi . 
sentence as words: 
 ['hi', '.']
--------------------------------------- starting preprocessing....
sentence:  Geh.
sentence lower cased and punctuation separated:
 geh . 
sentence as words: 
 ['geh', '.']
--------------------------------------- starting preprocessing....
sentence:  Hallo!
sentence lower cased and punctuation separated:
 hallo ! 
sentence as words: 
 ['hallo', '!']

In [48]:
sample = "are you alright"

print("\nBPE tokens:")

print(tokenizer.tokenize(sample))

print("\nToken IDs:")

print(tokenizer.numericalize(sample))



BPE tokens:
--------------------------------------- starting preprocessing....
sentence:  are you alright
sentence lower cased and punctuation separated:
 are you alright
sentence as words: 
 ['are', 'you', 'alright']
['a', 'r', 'e', '</w>', 'y', 'o', 'u', '</w>', 'a', 'l', 'r', 'i', 'g', 'h', 't', '</w>']

Token IDs:
--------------------------------------- starting preprocessing....
sentence:  are you alright
sentence lower cased and punctuation separated:
 are you alright
sentence as words: 
 ['are', 'you', 'alright']
[11, 14, 10, 4, 1, 6, 1, 4, 11, 12, 14, 9, 5, 8, 17, 4]


In [49]:
ids = tokenizer.numericalize(sample)

print("\nDecoded:")

print(tokenizer.decode(ids))


--------------------------------------- starting preprocessing....
sentence:  are you alright
sentence lower cased and punctuation separated:
 are you alright
sentence as words: 
 ['are', 'you', 'alright']

Decoded:
are <UNK>o<UNK> alright


In [50]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, tokenizer):

        self.pairs = pairs
        self.tokenizer = tokenizer

    def __len__(self):

        return len(self.pairs)

    def __getitem__(self, idx):

        src_sent, trg_sent = self.pairs[idx]

        src_tokens = self.tokenizer.numericalize(src_sent)

        trg_tokens = self.tokenizer.numericalize(trg_sent)

        src_tensor = torch.tensor(
            src_tokens + [self.tokenizer.word2idx["<EOS>"]], dtype=torch.long
        )

        trg_tensor = torch.tensor(
            [self.tokenizer.word2idx["<SOS>"]]
            + trg_tokens
            + [self.tokenizer.word2idx["<EOS>"]],
            dtype=torch.long,
        )

        return (src_tensor, trg_tensor)


In [51]:
def collate_fn(batch):

    src_batch, trg_batch = zip(*batch)

    src_batch = pad_sequence(src_batch, padding_value=0, batch_first=True)

    trg_batch = pad_sequence(trg_batch, padding_value=0, batch_first=True)

    return (src_batch, trg_batch)


In [52]:
dataset = TranslationDataset(pairs, tokenizer)


dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)


src_batch, trg_batch = next(iter(dataloader))

print("\nSource batch shape:", src_batch.shape)

print("Target batch shape:", trg_batch.shape)

print("\nSource batch:")

print(src_batch)

print("\nTarget batch:")

print(trg_batch)

--------------------------------------- starting preprocessing....
sentence:  Go.
sentence lower cased and punctuation separated:
 go . 
sentence as words: 
 ['go', '.']
--------------------------------------- starting preprocessing....
sentence:  Geh.
sentence lower cased and punctuation separated:
 geh . 
sentence as words: 
 ['geh', '.']
--------------------------------------- starting preprocessing....
sentence:  Hi.
sentence lower cased and punctuation separated:
 hi . 
sentence as words: 
 ['hi', '.']
--------------------------------------- starting preprocessing....
sentence:  Grüß Gott!
sentence lower cased and punctuation separated:
 grüß gott ! 
sentence as words: 
 ['grüß', 'gott', '!']
--------------------------------------- starting preprocessing....
sentence:  Hi.
sentence lower cased and punctuation separated:
 hi . 
sentence as words: 
 ['hi', '.']
--------------------------------------- starting preprocessing....
sentence:  Hallo!
sentence lower cased and punctuation s

In [53]:
class EmbeddingLayer(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len=5000):
        print(
            "starting embedding layer inititialization ______________________________________________"
        )
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = d_model
        self.max_seq_len = max_seq_len
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.dropout = nn.Dropout(0.1)
        pos_enc = self._calculatePositionalEncoding2()
        self.register_buffer("pos_enc", pos_enc)

        print("vocab size: ", vocab_size)
        print("embedding dimension: ", self.emb_dim)
        print("max sequence length: ", max_seq_len)
        print("embedding : ", self.embedding)
        print("drop out layerL ", self.dropout)
        print("positional encoding: ", pos_enc.shape)
        print(
            "end embedding initialization -------------------------------------------------"
        )

    # src is 2D
    def _calculatePositionalEncoding1(self):
        if self.emb_dim % 2 != 0:
            raise ValueError("emb_dim must be even for positional encoding")

        pos_enc = torch.empty(self.max_seq_len, self.emb_dim)
        print("pos enc zeros shape: ", pos_enc.shape)
        for pos in range(self.max_seq_len):
            dim = 0
            while dim < self.emb_dim // 2:
                dim_param = (2 * dim) / self.emb_dim

                sinusoid_param = pos / math.pow(10000, dim_param)
                sin_pos_enc = math.sin(sinusoid_param)
                pos_enc[pos, 2 * dim] = sin_pos_enc
                cos_pos_enc = math.cos(sinusoid_param)
                pos_enc[pos, 2 * dim + 1] = cos_pos_enc
                dim += 1

        # positional_tensor = pos_enc.unsqueeze(0)

        return pos_enc

    def _calculatePositionalEncoding2(self):
        print(
            "_____________________starting _calculatePositionalEncoding2 _____________________________________"
        )
        if self.emb_dim % 2 != 0:
            raise ValueError("emb_dim must be even for positional encoding")

        pos = torch.arange(self.max_seq_len, dtype=torch.float32)
        print("positions encoding shape: ", pos.shape)
        dim_indices = torch.arange(0, self.emb_dim, 2, dtype=torch.float32)
        print("print dim indices shape: ", dim_indices.shape)
        denom = torch.pow(10000.0, dim_indices / self.emb_dim)
        print("denom shape: ", denom.shape)
        angles = pos.unsqueeze(1) / denom.unsqueeze(0)
        print("angles shape: ", angles.shape)
        pos_enc = torch.zeros(self.max_seq_len, self.emb_dim)
        print("pos enc zeros shape before updating: ", pos_enc)
        pos_enc[:, 0::2] = torch.sin(angles)
        pos_enc[:, 1::2] = torch.cos(angles)
        print("pos_enc updated with cos and sin values")
        # positional_tensor = pos_enc.unsqueeze(0)
        print(
            "-------------------------end _calculatePositionalEncoding2 ---------------------------"
        )
        return pos_enc

    def forward(self, src):
        print(
            "___________________embedding forward starting _____________________________________________"
        )
        batch_size, seq_len = src.shape
        print("src shape:(batch_size, seq_len): ", src.shape)
        if seq_len > self.max_seq_len:
            raise ValueError(
                f"Sequence length ({seq_len}) exceeds max_seq_len ({self.max_seq_len})"
            )

        embedded = self.embedding(src) * math.sqrt(self.emb_dim)
        print("embedding shape: ", embedded.shape)
        print("pos enc shape before unsqueeze: ", self.pos_enc)
        pos_enc = self.pos_enc.unsqueeze(0)
        print("pos enc shape after unsqueeze: ", pos_enc.shape)
        print(
            "cut out shape to be added to embeeding: pos_enc[:, :seq_len, :]",
            pos_enc[:, :seq_len, :].shape,
        )
        updated_embedded = embedded + pos_enc[:, :seq_len, :]
        print("updated_embedded: ", updated_embedded.shape)
        output_embedding = self.dropout(updated_embedded)
        print("output embedding: ", output_embedding.shape)
        print(
            "---------------------------end _calculatePositionalEncoding2 ----------------------------------------------------"
        )
        return output_embedding

In [54]:
random_data = torch.rand(2, 3, 3)
random_data_2d = torch.rand(3, 4)
print("mask: -- \n: ", random_data)
attn_shape = random_data.shape[-2:]
print("mask last shapes: \n", attn_shape)
ones = torch.ones(attn_shape, device=random_data.device)
print("ones --- :\n", ones)
upper_tri = torch.triu(ones, diagonal=1).bool()
print("upper tri: --\n", upper_tri)
scaled = random_data.masked_fill(upper_tri, float("-inf"))
print("scalled: -- \n", scaled)

mask: -- 
:  tensor([[[0.5952, 0.0405, 0.3004],
         [0.1060, 0.4067, 0.5498],
         [0.5260, 0.9248, 0.8686]],

        [[0.0706, 0.8852, 0.6381],
         [0.6497, 0.1299, 0.7850],
         [0.3512, 0.5988, 0.6985]]])
mask last shapes: 
 torch.Size([3, 3])
ones --- :
 tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
upper tri: --
 tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])
scalled: -- 
 tensor([[[0.5952,   -inf,   -inf],
         [0.1060, 0.4067,   -inf],
         [0.5260, 0.9248, 0.8686]],

        [[0.0706,   -inf,   -inf],
         [0.6497, 0.1299,   -inf],
         [0.3512, 0.5988, 0.6985]]])


In [55]:
class AttentionModel(nn.Module):
    def __init__(self, d_model, d_k, d_v):
        print(
            "__________________starting AttentionModel initialization______________________________"
        )
        super().__init__()
        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_v
        self.Wq = nn.Linear(d_model, d_k)
        self.Wk = nn.Linear(d_model, d_k)
        self.Wv = nn.Linear(d_model, d_v)
        print("d_model: ", d_model)
        print("d_k: ", d_k)
        print("d_v: ", d_v)
        print("weights W_q: ", self.Wq)
        print("weights W_k: ", self.Wk)
        print("weights W_v: ", self.Wv)
        print(
            "---------------end AttentionModel initialization -------------------------------------"
        )

    def forward(
        self, query_source, key_source, value_source, padd_mask=None, mask=False
    ):
        print(
            "_____________ AttentionModel forward starting ________________________________"
        )
        print("query source: ", query_source.shape)
        print("key source: ", key_source.shape)
        print("value source: ", value_source.shape)
        print("mask, apply upper triangular mask :", mask)
        if padd_mask is not None:
            print("padd mask shape: ", padd_mask.shape)
        K = self.Wk(key_source)
        V = self.Wv(value_source)
        Q = self.Wq(query_source)

        print("V: ", V.shape)
        print("K: ", K.shape)
        print("Q: ", Q.shape)

        K_t = K.transpose(-2, -1)
        print("K transpose: ", K_t.shape)

        # QK = torch.bmm(Q, K_t)
        QK = torch.matmul(Q, K_t)
        print("Q*K: ", QK.shape)
        scaled = QK / math.sqrt(self.d_k)

        batch_num, row_num, column_num = scaled.shape

        # if mask:
        #     for batch in range(batch_num):
        #         for word_index in range(row_num):
        #             for dim_index in range(column_num):
        #                 if dim_index > word_index:
        #                     scaled[batch, word_index, dim_index] = -float("inf")

        if mask:
            attn_shape = scaled.shape[-2:]
            print("attn_shape: ", attn_shape)
            upper_tri = torch.triu(
                torch.ones(attn_shape, device=scaled.device), diagonal=1
            ).bool()
            print("upper tri: ", upper_tri)
            print("upper tri shape: ", upper_tri.shape)

            scaled = scaled.masked_fill(upper_tri, float("-inf"))
            print("scaled: ", scaled.shape)
        # pad mask is 2d(batch,seq_len) , while scale is 3d
        if padd_mask is not None:
            print("padd mask before unsqueeze: ", padd_mask.shape)
            padd_mask = padd_mask.unsqueeze(1)
            print("pad mask after squeeze: ", padd_mask.shape)
            scaled = scaled.masked_fill(padd_mask, float("-inf"))

        normalized = F.softmax(scaled, dim=-1)
        print("normalized : ", normalized.shape)
        # attention = torch.bmm(normalized, V)
        attention = torch.matmul(normalized, V)
        print("attention: ", attention.shape)
        print(
            "----------------------end AttentionModel forward ----------------------------------------"
        )
        return attention


class MultiHeadAttentionModel(nn.Module):
    def __init__(self, h, d_model, d_k, d_v):
        print(
            "_____________________ start MultiHeadAttentionModel initialiazation __________________"
        )
        super().__init__()
        print("h: ", h)
        print("d_model: ", d_model)
        print("d_k: ", d_k)
        print("d_v", d_v)
        self.heads = nn.ModuleList()
        print("heads before adding models: ", self.heads)
        self.feedforward = nn.Linear(h * d_v, d_model)
        print("feedfoward: ", self.feedforward)
        self.h = h

        for _ in range(h):
            layer = AttentionModel(d_model, d_k, d_v)
            self.heads.append(layer)
        # self.attention_head = attention_head
        print("heads after adding models: ", self.heads)
        print(
            "----------------------end MultiHeadAttentionModel initialization ---------------------------"
        )

    def forward(
        self,
        query_source,
        key_source,
        value_source,
        mask=False,
        padd_mask=None,
    ):
        print(
            "____________________ start MultiHeadAttentionModel forward_________________________________"
        )
        print("query source: ", query_source.shape)
        print("key source: ", key_source.shape)
        print("value source: ", value_source.shape)
        print("mask, apply upper triangular mask :", mask)
        all_heads = []
        for layer in self.heads:
            attention = layer(
                query_source=query_source,
                key_source=key_source,
                value_source=value_source,
                mask=mask,
                padd_mask=padd_mask,
            )
            all_heads.append(attention)
        print("all heads: ", len(all_heads))
        concat_heads = torch.cat(all_heads, dim=-1)
        print("concat heads: ", concat_heads.shape)
        feedforward = self.feedforward(concat_heads)
        print("feedforward: ", feedforward.shape)
        print(
            "----------------------- end MultiHeadAttentionModel forward ------------------------------------"
        )
        return feedforward


In [56]:
class AttentionSublayer(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, dropout=0.1):
        print(
            "________________starting AttentionSublayer initialization _____________________________________"
        )
        super().__init__()
        self.multi_head_attention_sublayer = MultiHeadAttentionModel(
            h, d_model, d_k, d_v
        )
        print("multihead_attention: ", self.multi_head_attention_sublayer)
        self.normalization = nn.LayerNorm(d_model)
        print("nromalization :", self.normalization)
        self.dropout = nn.Dropout(dropout)
        print("drop out: ", self.dropout)
        print(
            "-------------------------end AttentionSublayer initialization-----------------------------"
        )

    def forward(
        self, src, query_source, key_source, value_source, mask=False, padd_mask=None
    ):
        print(
            "________________starting AttentionSublayer forward _____________________________________"
        )
        attention_output = self.multi_head_attention_sublayer(
            query_source=query_source,
            key_source=key_source,
            value_source=value_source,
            mask=mask,
            padd_mask=padd_mask,
        )
        print("attention output: ", attention_output.shape)
        add_dropout = self.dropout(attention_output)
        print("add dropout: ", add_dropout.shape)
        add_residual = src + add_dropout
        print("add_residual :", add_residual.shape)
        output = self.normalization(add_residual)
        print("output: ", output.shape)
        print(
            "------------------------- end AttentionSublayer forward -----------------------------"
        )
        return output


In [57]:
class FeedForwardSublayer(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        print(
            "________________starting FeedForwardSublayer initialization_____________________________________"
        )
        super().__init__()
        print("d_model: ", d_model)

        if d_ff is None:
            d_ff = 4 * d_model
        print("dff: ", d_ff)
        self.feedforward1 = nn.Linear(d_model, d_ff)
        print("feedforward1: ", self.feedforward1)
        self.feedforward2 = nn.Linear(d_ff, d_model)
        print("feedforward2: ", self.feedforward2)
        self.normalization = nn.LayerNorm(d_model)
        print("normalization: ", self.normalization)
        self.dropout = nn.Dropout(dropout)
        print("dropout: ", self.dropout)
        self.relu = nn.ReLU()
        print("relu: ", self.relu)
        print(
            "------------------------- end FeedForwardSublayer initialization -----------------------------"
        )

    def forward(self, src):
        print(
            "________________starting FeedForwardSublayer forward_____________________________________"
        )
        forward1 = self.feedforward1(src)
        print("forward1: ", forward1.shape)
        activated1 = self.relu(forward1)
        print("activated1: ", activated1.shape)
        forward2 = self.feedforward2(activated1)
        print("forward2: ", forward2.shape)
        add_dropout = self.dropout(forward2)
        print("add_dropout: ", add_dropout.shape)
        add_residual = src + add_dropout
        print("add_residual: ", add_residual.shape)
        output = self.normalization(add_residual)
        print("output: ", output.shape)
        print(
            "------------------------- end FeedForwardSublayer forward -----------------------------"
        )

        return output


In [58]:
class Encoder(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, embedding, n_layers=6):
        print(
            "________________starting Encoder initialization_____________________________________"
        )
        super().__init__()
        self.embedding_layer = embedding
        print("embedding: ", embedding)
        self.layers = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        "attention": AttentionSublayer(
                            h=h, d_model=d_model, d_k=d_k, d_v=d_v
                        ),
                        "feed_forward": FeedForwardSublayer(d_model=d_model),
                    }
                )
                for layer in range(n_layers)
            ]
        )
        print("layers: ", self.layers)
        print(
            "------------------------- end Encoder initialization -----------------------------"
        )

    def forward(self, src):
        print(
            "________________starting Encoder forward _____________________________________"
        )
        padd_mask = src == 0
        print("src: ", src.shape)
        encoded = self.embedding_layer(src)
        print("encoded: ", encoded.shape)
        print("starting loop for attention layers")
        for layer in self.layers:
            attention = layer["attention"](
                src=encoded,
                query_source=encoded,
                key_source=encoded,
                value_source=encoded,
                padd_mask=padd_mask,
            )
            feedforward_output = layer["feed_forward"](attention)
            encoded = feedforward_output

        print(
            "------------------------- end Encoder forward -----------------------------"
        )
        return encoded

In [59]:
class Decoder(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, embedding, n_layers=6):
        print(
            "________________starting Decoder initialization_____________________________________"
        )
        super().__init__()
        print("h: ", h)
        print("d_model: ", d_model)
        print("d_k: ", d_k)
        print("d_v", d_v)
        print("embedding: ", embedding)
        self.embedding_layer = embedding

        self.layers = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        "self_attention": AttentionSublayer(h, d_model, d_k, d_v),
                        "cross_attention": AttentionSublayer(h, d_model, d_k, d_v),
                        "feed_forward": FeedForwardSublayer(d_model),
                    }
                )
                for layers in range(n_layers)
            ]
        )
        print("attention layers: ", self.layers)
        print(
            "------------------------- end Decoder initialization -----------------------------"
        )

    def forward(self, src, encoder_output, src_padd_mask=None):
        print(
            "________________starting Decoder forward _____________________________________"
        )
        trg_padd_mask = src == 0
        print("src: ", src.shape)
        print("encoder output: ", encoder_output.shape)
        decoded = self.embedding_layer(src)
        print("decoded: ", decoded.shape)
        print("starting to loop through attention layers")
        for layer in self.layers:
            self_attention = layer["self_attention"](
                src=decoded,
                query_source=decoded,
                key_source=decoded,
                value_source=decoded,
                mask=True,
                padd_mask=trg_padd_mask,
            )
            cross_attention = layer["cross_attention"](
                src=self_attention,
                query_source=self_attention,
                key_source=encoder_output,
                value_source=encoder_output,
                mask=False,
                padd_mask=src_padd_mask,
            )
            feedforward_output = layer["feed_forward"](cross_attention)
            decoded = feedforward_output
        print("decoded: ", decoded.shape)
        output = torch.matmul(decoded, self.embedding_layer.embedding.weight.T)
        print("output: ", output.shape)
        print(
            "------------------------- end Decoder forward -----------------------------"
        )
        return output


In [60]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        print(
            "____________________start Seq2Seq initialization________________________________"
        )
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        print("encoder: ", encoder)
        print("decoder: ", decoder)
        print("device: ", device)
        print(
            "------------------------- end Seq2Seq initialization -----------------------------"
        )

    def forward(
        self,
        src,
        trg,
        # vocab_size,
    ):
        print(
            "____________________start Seq2Seq forward ________________________________"
        )
        print("src :", src.shape)
        print("trg: ", trg.shape)
        src_padd_mask = src == 0

        # batch_size = src.shape[0]
        # trg_len = trg.shape[1]

        # outputs = torch.zeros(batch_size, trg_len, vocab_size).to(self.device)
        encoder_output = self.encoder(src)
        print("encoder output: ", encoder_output.shape)
        decoder_output = self.decoder(trg, encoder_output, src_padd_mask)
        print("decoder output:", decoder_output.shape)
        print(
            "------------------------- end Seq2Seq forward -----------------------------"
        )
        return decoder_output

In [61]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = tokenizer.num_words
D_MODEL = 512
D_V = 64
D_K = 64
H = 8

embedding_layer = EmbeddingLayer(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, max_seq_len=5000
)
enc = Encoder(h=H, d_model=D_MODEL, d_k=D_K, d_v=D_V, embedding=embedding_layer)
dec = Decoder(h=H, d_model=D_MODEL, d_k=D_K, d_v=D_V, embedding=embedding_layer)
model = Seq2Seq(encoder=enc, decoder=dec, device=device).to(device)

beta_1 = 0.9
beta_2 = 0.98
epsilon = 1e-9
label_smoothing = 0.1
N_EPOCHS = 10

total_steps = N_EPOCHS * len(dataloader)

warmup_steps = 0.4 * total_steps


optimizer = torch.optim.Adam(
    model.parameters(),
    betas=(beta_1, beta_2),
    eps=epsilon,
    lr=1.0,
)


def getLearningRate(step):
    step_num = step + 1
    lrate = (D_MODEL ** (-0.5)) * min(
        (step_num ** (-0.5)), (step_num * (warmup_steps ** (-1.5)))
    )
    return lrate


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=getLearningRate)

criterion = nn.CrossEntropyLoss(
    ignore_index=0,
    label_smoothing=label_smoothing,
)  # Ignore <PAD> tokens in loss calculation

print(f"Model initialized successfully. Device: {device}")

starting embedding layer inititialization ______________________________________________
_____________________starting _calculatePositionalEncoding2 _____________________________________
positions encoding shape:  torch.Size([5000])
print dim indices shape:  torch.Size([256])
denom shape:  torch.Size([256])
angles shape:  torch.Size([5000, 256])
pos enc zeros shape before updating:  tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
pos_enc updated with cos and sin values
-------------------------end _calculatePositionalEncoding2 ---------------------------
vocab size:  39
embedding dimension:  512
max sequence length:  5000
embedding :  Embedding(39, 512)
drop out layerL  Dropout(p=0.1, inplace=False)
positional encoding:  torch.Size([5000, 512])
end embedding initialization ----------

In [62]:
def train(model, dataloader, optimizer, criterion):
    print("________________starting training _____________________________")

    model.train()
    epoch_loss = 0
    print("starting training loop")
    for i, (src, trg) in enumerate(dataloader):
        src = src.to(device)
        trg = trg.to(device)
        print("src: ", src.shape)
        print("trg: ", trg.shape)

        optimizer.zero_grad()
        output = model(src, trg[:, :-1])
        print("output: ", output.shape)
        flat_trg = trg[:, 1:].reshape(-1)
        print("flat_trg: ", flat_trg.shape)
        flat_output = output.reshape(-1, VOCAB_SIZE)
        print("flat_output: ", flat_output.shape)

        loss = criterion(flat_output, flat_trg)
        print("loss: ", loss)
        loss.backward()

        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()
    print("------------------- end training ---------------------------")
    return epoch_loss / len(dataloader)


for epoch in range(N_EPOCHS):
    start_time = time.time()

    train_loss = train(model, dataloader, optimizer, criterion)
    end_time = time.time()

    print(f"Epoch: {epoch + 1:02} | Time: {end_time - start_time:.2f}s")
    print(
        f"\tTrain Loss: {train_loss:.3f} | Jarring Perplexity: {math.exp(train_loss):.3f}"
    )

________________starting training _____________________________
starting training loop
--------------------------------------- starting preprocessing....
sentence:  Hi.
sentence lower cased and punctuation separated:
 hi . 
sentence as words: 
 ['hi', '.']
--------------------------------------- starting preprocessing....
sentence:  Hallo!
sentence lower cased and punctuation separated:
 hallo ! 
sentence as words: 
 ['hallo', '!']
--------------------------------------- starting preprocessing....
sentence:  Hi.
sentence lower cased and punctuation separated:
 hi . 
sentence as words: 
 ['hi', '.']
--------------------------------------- starting preprocessing....
sentence:  Grüß Gott!
sentence lower cased and punctuation separated:
 grüß gott ! 
sentence as words: 
 ['grüß', 'gott', '!']
--------------------------------------- starting preprocessing....
sentence:  Go.
sentence lower cased and punctuation separated:
 go . 
sentence as words: 
 ['go', '.']
------------------------------